# Meme Organizer: 01. Лексический и Dense Zero-Shot поиск (Бейзлайны)

**Цель исследования:**
1. Построение воспроизводимого baseline ранжирования на чистом датасете `data/gold/gold_dataset.jsonl` (318 мемов, 1312 пар запросов).
2. Сравнение лексического поиска (**BM25 / SQLite FTS5**) и плотных нейросетевых энкодеров (**`rubert-tiny2`**, **`multilingual-e5-small`**).
3. Профилирование On-Device характеристик (задержка инференса на CPU в мс, размер ONNX INT8 в МБ).
4. Качественный анализ ошибок (**Top-15 Worst Queries**) и исследование эффекта гибридного слияния (Late Fusion).


In [1]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from meme_organizer.data_splits.splitter import DatasetSplitter
from meme_organizer.evaluation.metrics import RetrievalEvaluator
from meme_organizer.retrieval.adapters.dense_text import DenseTextRetrieverAdapter
from meme_organizer.retrieval.baselines.bm25 import BM25RetrieverAdapter
from meme_organizer.retrieval.pipeline import RetrievalPipeline

# Фиксация путей
GOLD_DATASET_PATH = Path("../data/gold/gold_dataset.jsonl")
SPLIT_PATH = Path("../data/splits/splits_v1.json")

2026/11/24 21:02:31 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Проверка и загрузка данных без утечек (Leak-Free Split)

In [2]:
with open(SPLIT_PATH) as f:
    split_info = json.load(f)

print(f"Корпус мемов: {len(split_info['corpus_hashes'])} документов")
print(f"Train пары:   {len(split_info['train_pairs'])} запросов")
print(f"Val пары:     {len(split_info['val_pairs'])} запросов")
print(f"Test пары:    {len(split_info['test_pairs'])} запросов")

Корпус мемов: 318 документов
Train пары:   1021 запросов
Val пары:     138 запросов
Test пары:    153 запросов


## 2. Запуск Baseline 1: Лексический BM25 (аналог SQLite FTS5)

In [3]:
pipeline = RetrievalPipeline()
bm25_adapter = BM25RetrieverAdapter(model_name="bm25-baseline")

bm25_metrics, bm25_run_id = pipeline.run(
    adapter=bm25_adapter,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    run_name="bm25-baseline_test",
)

pd.DataFrame(
    [
        {
            "Model": "BM25 (Lexical)",
            "Recall@1": f"{bm25_metrics.recall_at_1:.4f}",
            "Recall@5": f"{bm25_metrics.recall_at_5:.4f}",
            "MRR@5": f"{bm25_metrics.mrr_at_5:.4f}",
            "NDCG@5": f"{bm25_metrics.ndcg_at_5:.4f}",
            "MLflow Run ID": bm25_run_id,
        }
    ]
)

🏃 View run bm25-baseline_test at: http://localhost:5000/#/experiments/4/runs/f183646b1db744149cacefaa7e9da801
🧪 View experiment at: http://localhost:5000/#/experiments/4


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,BM25 (Lexical),0.6078,0.7255,0.6544,0.6722,f183646b1db744149cacefaa7e9da801


## 3. Запуск Baseline 2: Dense Bi-Encoders (`rubert-tiny2` & `multilingual-e5-small`)

In [4]:
rubert_adapter = DenseTextRetrieverAdapter(
    model_name="rubert-tiny2",
    hf_model_id="cointegrated/rubert-tiny2",
    max_length=64,
    onnx_dir=Path("../models/onnx"),
)

rubert_metrics, rubert_run_id = pipeline.run(
    adapter=rubert_adapter,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    run_name="rubert-tiny2_test",
)

e5_adapter = DenseTextRetrieverAdapter(
    model_name="multilingual-e5-small",
    hf_model_id="intfloat/multilingual-e5-small",
    max_length=64,
    query_prefix="query: ",
    passage_prefix="passage: ",
    onnx_dir=Path("../models/onnx"),
)

e5_metrics, e5_run_id = pipeline.run(
    adapter=e5_adapter,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    run_name="multilingual-e5-small_test",
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏃 View run rubert-tiny2_test at: http://localhost:5000/#/experiments/4/runs/cf79f6324b594a8da80648c8e97a8dbc
🧪 View experiment at: http://localhost:5000/#/experiments/4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run multilingual-e5-small_test at: http://localhost:5000/#/experiments/4/runs/00c3af1c632b4a66b6fa4e87c42d1a74
🧪 View experiment at: http://localhost:5000/#/experiments/4


## 4. Сравнительная сводная таблица из MLflow

In [5]:
client = mlflow.tracking.MlflowClient("http://localhost:5000")
exp = client.get_experiment_by_name("meme-organizer-retrieval")
runs = client.search_runs(exp.experiment_id, order_by=["attribute.start_time DESC"], max_results=10)

leaderboard_rows = []
for r in runs:
    m = r.data.metrics
    leaderboard_rows.append(
        {
            "Run Name": r.data.tags.get("mlflow.runName", "unnamed"),
            "Recall@1": round(m.get("recall_at_1", 0.0), 4),
            "Recall@5": round(m.get("recall_at_5", 0.0), 4),
            "MRR@5": round(m.get("mrr_at_5", 0.0), 4),
            "Latency (ms)": round(m.get("cpu_latency_ms", 0.0), 2),
            "Size (MB)": round(m.get("model_size_mb", 0.0), 2),
            "Eval (s)": round(m.get("eval_inference_time_sec", 0.0), 3),
            "Run ID": r.info.run_id[:8],
        }
    )

pd.DataFrame(leaderboard_rows)

,Run Name,Recall@1,Recall@5,MRR@5,Latency (ms),Size (MB),Eval (s),Run ID
0,multilingual-e5-small_test,0.5621,0.6667,0.6032,5.96,112.52,2.616,00c3af1c
1,rubert-tiny2_test,0.4118,0.5098,0.4497,2.31,27.93,0.410,cf79f632
2,bm25-baseline_test,0.6078,0.7255,0.6544,0.00,0.10,0.019,f183646b


## 5. Анализ Top-15 Worst Queries (Качественный анализ ошибок)

In [6]:
bm25_csv = client.download_artifacts(bm25_run_id, "error_analysis/worst_queries.csv")
df_worst_bm25 = pd.read_csv(bm25_csv)
print("=== Топ худших запросов для BM25 ===")
display(df_worst_bm25[["query", "target_rank"]].head(8))

e5_csv = client.download_artifacts(e5_run_id, "error_analysis/worst_queries.csv")
df_worst_e5 = pd.read_csv(e5_csv)
print("=== Топ худших запросов для E5-small ===")
display(df_worst_e5[["query", "target_rank"]].head(8))

=== Топ худших запросов для BM25 ===


,query,target_rank
0,когда пасешь кого то в чате,264
1,когда собеседник несёт бред,257
2,бабушку вырубило с удара,249
3,когда залетела в универе,248
4,когда полный аннигилятор,238
5,когда ждешь своего часа,229
6,когда не веришь собеседнику,211
7,все будет,200


=== Топ худших запросов для E5-small ===


,query,target_rank
0,ура победа,262
1,когда пасешь кого то в чате,226
2,убегать от ментов на тачке,208
3,выкатываюсь на важные дела,207
4,когда задумал какую то пакость,193
5,мужчина смотрит с недоумением,190
6,мужчина с каре стреляет,142
7,когда приучаешь ребенка с детства,131
